# Ch.4 — Backpropagation calculus

*3Blue1Brown, Deep Learning series, Ch.4. Concise lecture notes. Ch.3 gave the **intuition**; this chapter makes it **formal** — the chain rule, the way ML people actually write it.*

**Goal:** for each weight & bias, find how **sensitive** the cost is to it — i.e. the partial derivative — since those partials *are* the gradient $\nabla C$. Expect some confusion; pause & ponder.

## The toy network: one neuron per layer

Strip it to the bone: every layer has a **single** neuron → 3 weights, 3 biases. Focus on the **last connection**.

**Notation** (superscripts $^{(L)}$ are layer *indices*, not exponents):

| Symbol | Meaning |
|---|---|
| $a^{(L)}$ | activation of the last neuron |
| $a^{(L-1)}$ | activation of the previous neuron |
| $w^{(L)},\ b^{(L)}$ | weight & bias on the last connection |
| $y$ | desired output for this training example (e.g. 0 or 1) |
| $z^{(L)}$ | the **weighted sum** (pre-activation) |

**Forward chain for one example:**

$$z^{(L)} = w^{(L)} a^{(L-1)} + b^{(L)}, \qquad a^{(L)} = \sigma\!\left(z^{(L)}\right), \qquad C_0 = \left(a^{(L)} - y\right)^2$$

Conceptually: $w, a^{(L-1)}, b \rightarrow z \rightarrow a \rightarrow C_0$. Each is just a number on its own little number line.

## The chain rule for $\partial C_0/\partial w^{(L)}$

A tiny nudge $\partial w^{(L)}$ nudges $z^{(L)}$, which nudges $a^{(L)}$, which nudges $C_0$. Multiply the three ratios:

$$\frac{\partial C_0}{\partial w^{(L)}} = \frac{\partial z^{(L)}}{\partial w^{(L)}}\,\frac{\partial a^{(L)}}{\partial z^{(L)}}\,\frac{\partial C_0}{\partial a^{(L)}}$$

**The three pieces:**

$$\frac{\partial C_0}{\partial a^{(L)}} = 2\left(a^{(L)} - y\right) \qquad \frac{\partial a^{(L)}}{\partial z^{(L)}} = \sigma'\!\left(z^{(L)}\right) \qquad \frac{\partial z^{(L)}}{\partial w^{(L)}} = a^{(L-1)}$$

**Read the meanings:**
- $2(a^{(L)}-y)$ → proportional to **how wrong** the output is; big error ⇒ big impact.
- $\sigma'(z^{(L)})$ → slope of the nonlinearity (sigmoid/ReLU).
- $a^{(L-1)}$ → a weight's influence scales with **how active the previous neuron is** → *"neurons that fire together, wire together."*

## From one example to the gradient

- The above is for **one** training example. Full cost = **average** over all examples, so:

$$\frac{\partial C}{\partial w^{(L)}} = \frac{1}{N}\sum_{k=1}^{N}\frac{\partial C_k}{\partial w^{(L)}}$$

- This is just **one component** of $\nabla C$ — but computing it is **>50% of the work**; the rest reuse the same pieces.

## Bias and the backward step

**Bias:** identical chain, swap $\dfrac{\partial z}{\partial w}$ for $\dfrac{\partial z}{\partial b} = 1$:

$$\frac{\partial C_0}{\partial b^{(L)}} = \frac{\partial a^{(L)}}{\partial z^{(L)}}\,\frac{\partial C_0}{\partial a^{(L)}} = \sigma'\!\left(z^{(L)}\right)\cdot 2\left(a^{(L)}-y\right)$$

**Propagating backward — the key piece:** sensitivity of $z^{(L)}$ to the *previous activation* is the weight itself:

$$\frac{\partial z^{(L)}}{\partial a^{(L-1)}} = w^{(L)}$$

We can't set $a^{(L-1)}$ directly, but this lets us **iterate the same chain rule one layer back** → and back, and back. That recursion *is* backpropagation.

## Multiple neurons per layer (the real case)

Surprisingly little changes — just **more indices**. Index layer $L-1$ with $k$, layer $L$ with $j$.

- **Cost** sums over output neurons: $\displaystyle C_0 = \sum_{j}\left(a^{(L)}_j - y_j\right)^2$
- **Weight** $w^{(L)}_{jk}$ connects neuron $k$ (layer $L-1$) → neuron $j$ (layer $L$). *(Order $jk$ matches the weight-matrix convention from Ch.1.)*
- $z^{(L)}_j = \sum_k w^{(L)}_{jk}\,a^{(L-1)}_k + b^{(L)}_j$, and $a^{(L)}_j = \sigma(z^{(L)}_j)$.

The chain-rule expression for $\partial C_0/\partial w^{(L)}_{jk}$ looks **essentially the same** as the one-neuron case.

**What's genuinely different:** a neuron $a^{(L-1)}_k$ now feeds **every** neuron $j$ in the next layer, so it affects the cost through **multiple paths** — you must **sum over $j$**:

$$\frac{\partial C_0}{\partial a^{(L-1)}_k} = \sum_{j} \frac{\partial z^{(L)}_j}{\partial a^{(L-1)}_k}\,\frac{\partial a^{(L)}_j}{\partial z^{(L)}_j}\,\frac{\partial C_0}{\partial a^{(L)}_j}$$

Once you have this for layer $L-1$, **repeat the whole process** for the weights & biases feeding into it.

In [ ]:
import numpy as np

# Backprop on the toy 1-neuron-per-layer network: verify the chain rule numerically.
def sigmoid(z):  return 1/(1+np.exp(-z))
def dsigmoid(z): return sigmoid(z)*(1-sigmoid(z))

w, b, a_prev, y = 0.8, -0.5, 0.6, 1.0          # last-connection params + previous activation + target

z = w*a_prev + b                                # pre-activation
a = sigmoid(z)
C = (a - y)**2

# Analytic chain rule:  dC/dw = dz/dw * da/dz * dC/da
dC_da = 2*(a - y)
da_dz = dsigmoid(z)
dz_dw = a_prev
dC_dw = dz_dw * da_dz * dC_da
dC_db = 1     * da_dz * dC_da                    # dz/db = 1
dC_dap = w    * da_dz * dC_da                    # dz/da_prev = w  (the backward step)

# Numeric check via finite difference:
eps = 1e-6
num_dC_dw = (( (sigmoid((w+eps)*a_prev+b)-y)**2 ) - C) / eps
print(f"dC/dw analytic = {dC_dw:.6f}   numeric = {num_dC_dw:.6f}")
print(f"dC/db = {dC_db:.6f}   dC/da_prev (backprop signal) = {dC_dap:.6f}")

## The big picture

- These chain-rule expressions give **every component of $\nabla C$** — the derivatives gradient descent uses to step downhill.
- The whole algorithm = **chain rule applied recursively, layer by layer, backward**, reusing each layer's result for the one before it.
- That's backpropagation — the workhorse behind how neural networks learn. *Don't worry if it takes time to digest.*

**Series complete:** Ch.1 structure → Ch.2 gradient descent → Ch.3 backprop intuition → **Ch.4 backprop calculus.**

**References**
- 3B1B **Essence of Calculus** series — for the chain rule itself.
- Michael Nielsen, *Neural Networks and Deep Learning* — backprop chapter & equations.